In [ ]:
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader

In [ ]:
df = pd.read_excel("step3result copy 2.xlsx") 
df

In [ ]:
categories = df['category'].unique()
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [10]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import DataFrameLoader

# remove this later
CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2024-02-15-preview",
    azure_deployment="ChatGPT4128k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-4",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


# CATEGORIZATION_SUMMARY_SYSTEM_TEMPLATE = """I am considering a scoping reivew project for given category I want to understand how the existing literature guides. Write a single detailed summary based on the full texts of the article for each category."""

SUMMARIZE_CATEGORY_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. All of the article summaries below were assigned the category {category}. Write a single paragrph final summary of the following journal article summaries, focusing on my question. Use APA-style in-text citations throughout the paragraph. The article summaries are separated by '---'"

SUMMARIZE_ARTICLE_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. The article below was assigned the category {category}. Write a single paragrph summarizing the artcle, focusing on my question. The entire paragraph will be cited using only the article being summarized. Write the paragaraph so providing only the citation for the single article will be appropriate."



HUMAN_TEMPLATE = """
Content to summarize:
{content}
"""

category_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_CATEGORY_TEMPLATE)
article_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_ARTICLE_TEMPLATE)

human_message_prompt = HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)

category_summary_chat_prompt = ChatPromptTemplate.from_messages([category_system_message_prompt, human_message_prompt])
article_summary_chat_prompt = ChatPromptTemplate.from_messages([article_system_message_prompt, human_message_prompt])


In [12]:
# use abtract when text is not available.
df['Text'] = df.apply(lambda row: row['abstract'] if row['Text'] == 'Text not available' else row['Text'], axis=1)

for current_category in categories:
    filtered_rows = df[(df['category'] == current_category)]
    # if not filtered_rows.empty:
    #     new_df = pd.concat([new_df, filtered_rows])
    article_summaries = []
    for _, row in filtered_rows.iterrows():
        result = CHAT.invoke(article_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content="APA Citation: " + row.APA_Citation + "\n\nArticle text:\n\n" +row.Text,
        ).to_messages())
        # TODO: double check citation column name
        summary_to_keep = f"APA Citation: {row.APA_Citation}\n\n Summary: {result.content}\n\n --- "
        print(summary_to_keep)
        article_summaries.append(summary_to_keep)

    text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=text_to_summarize
        ).to_messages())
    print(result.content)
    print("************")
    

APA Citation: Wijayanayaka, S., Guha, A., Sivanesan, K., & Veerasingham, M. (2021). Extra-axial haemorrhage in a patient with Alport syndrome after epidural anaesthesia. BMJ Case Reports CP, 14(6), e242160.

 Summary: In the case report by Wijayanayaka et al. (2021), an 18-year-old woman with Alport syndrome experienced an extra-axial hemorrhage (both extradural and subdural) following epidural anesthesia during a ventouse delivery. Despite the rarity of such complications, the patient's postpartum period was marked by a tonic-clonic seizure and severe headache, which were initially managed as a suspected post-dural puncture headache (PDPH). However, subsequent imaging revealed the hemorrhage, prompting a multidisciplinary approach to care, including neurosurgical consultation. The patient's condition was managed conservatively, with the hemorrhage remaining stable and the patient showing clinical improvement. This case underscores the importance of considering rare but serious complic